# 깃헙 PR 데이터 수집 API

실제 유명한 상용 오픈소스에 대해 Github 데이터를 수집한다.

예: Kubernetes


In [0]:
%pip install azure-keyvault-secrets azure-identity httpx
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Key Vault 접근하여 비빌 키 가져오기.

# <사전 준비>
# 0. Access Connector 준비
# 1. Credential은 Unity Catalog에서 생성
# 2. Access Connector 연결 및 Credential 연결 생성
# 3. Key Vault에서 Access Connector에 대한 Azure KeyVault 권한 IAM 설정
#########################################################################
from azure.keyvault.secrets import SecretClient 

credential = dbutils.credentials.getServiceCredentialsProvider(
  '2dtfin-gitjabi-access-conector-credential'
)
vault_url = "https://gitjabi-keyvault.vault.azure.net/"
secret_client = SecretClient(vault_url=vault_url, credential=credential)

#######################################################################
#   Github Token 발급은 Settings -> Developer Settings -> 
#     Fine Grained Access Token 기능 에서 발급
#   
#   내부 공용 목적으로 Organization 명의로 토큰 발급 했으니, 토큰은 이O건에게 문의.
#######################################################################   

GITHUB_TOKEN = secret_client.get_secret("github-token").value.strip()


## 데이터 수집 기본 모듈

In [0]:
import httpx, asyncio
import time
from typing import List, Dict
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


class GitHubConfig:
    """
    GitHub API 설정 및 인증 정보 관리
    """

    def __init__(self, github_token: str):
        self.github_token = github_token.strip()

    @property
    def headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"Bearer {self.github_token}",
            "Accept": "application/vnd.github+json"
        }

class GitHubPRFetcher:
    """
    GitHub Pull Request 수집기
    - Rate limit 보호
    - 페이지네이션 처리
    """
    GET_PULLS_BASE_URL = "https://api.github.com/repos/{repo_name}/{repo_name}/pulls"

    def __init__(
        self,
        config,
        repo_name: str,
        timeout: float = 20.0,
        max_concurrency: int = 5,
    ):
        self.config = config
        self.timeout = timeout
        self.max_concurrency = max_concurrency
        self._repo_name = repo_name

    async def _fetch_page(
        self,
        client: httpx.AsyncClient,
        semaphore: asyncio.Semaphore,
        state: str,
        per_page: int,
        page: int
    ) -> List[Dict]:

        async with semaphore:
            if page % 10 == 0:
                print(f"fetching: {page} pages over...")

            params = {
                "state": state,
                "per_page": per_page,
                "page": page
            }

            resp = await client.get(
                self.GET_PULLS_BASE_URL.format(repo_name="kubernetes"),
                params=params
            )

            # Rate limit 보호
            if resp.status_code == 403:
                reset = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
                sleep_sec = max(reset - time.time(), 10)
                print(f"[WARN] Rate limit hit. Sleeping {sleep_sec:.0f}s")
                await asyncio.sleep(sleep_sec)
                return []

            resp.raise_for_status()
            return resp.json()

    async def fetch_prs(
        self,
        state: str = "closed",
        max_pages: int = 10,
        per_page: int = 100
    ) -> List[Dict]:

        semaphore = asyncio.Semaphore(self.max_concurrency)
        prs: List[Dict] = []

        async with httpx.AsyncClient(
            timeout=self.timeout,
            headers=self.config.headers
        ) as client:

            tasks = [
                self._fetch_page(
                    client=client,
                    semaphore=semaphore,
                    state=state,
                    per_page=per_page,
                    page=page
                )
                for page in range(1, max_pages + 1)
            ]

            results = await asyncio.gather(*tasks)

        # flatten
        for batch in results:
            if batch:
                prs.extend(batch)

        return prs

    @staticmethod
    def _fetch_issue_comments(pr_number: int, headers: dict) -> list:
        url = f"https://api.github.com/repos/kubernetes/kubernetes/issues/{pr_number}/comments"
        results = []

        with httpx.Client(headers=headers, timeout=20) as client:
            page = 1
            while True:
                resp = client.get(url, params={"per_page": 100, "page": page})
                resp.raise_for_status()
                batch = resp.json()
                if not batch:
                    break

                for c in batch:
                    results.append({
                        "pr_number": pr_number,
                        "comment_id": c["id"],
                        "comment_type": "issue_comment",
                        "author": c["user"]["login"],
                        "body": c["body"],
                        "state": None,
                        "created_at": c["created_at"],
                        "updated_at": c["updated_at"],
                        "path": None,
                        "position": None,
                        "commit_id": None,
                        "raw": c
                    })
                page += 1

        return results

    @staticmethod
    def _fetch_reviews(pr_number: int, headers: dict) -> list:
        url = f"https://api.github.com/repos/kubernetes/kubernetes/pulls/{pr_number}/reviews"
        results = []

        with httpx.Client(headers=headers, timeout=20) as client:
            resp = client.get(url, params={"per_page": 100})
            resp.raise_for_status()

            for r in resp.json():
                results.append({
                    "pr_number": pr_number,
                    "comment_id": r["id"],
                    "comment_type": "review",
                    "author": r["user"]["login"],
                    "body": r.get("body"),
                    "state": r.get("state"),
                    "created_at": r["submitted_at"],
                    "updated_at": r["submitted_at"],
                    "path": None,
                    "position": None,
                    "commit_id": r.get("commit_id"),
                    "raw": r
                })

        return results


github_api_config = GitHubConfig(GITHUB_TOKEN)



## 유의미한 데이터/컬럼 전처리

### 실제 PR 데이터 예시

**Kubernetes 깃헙 PR**
https://github.com/kubernetes/kubernetes/pulls?q=is%3Apr+is%3Aclosed

```json
{
  "id": 3263340905,
  "number": 136885,
  "state": "closed",
  "title": "Add CertificateSigningRequest create log entry for default kubelet debug level V2",
  "html_url": "https://github.com/kubernetes/kubernetes/pull/136885",
  "user": {
    "login": "youwalther65",
    "type": "User",
    "site_admin": false
  },
  "body": "#### What type of PR is this?\r\n/kind feature\r\n\r\n#### What this PR does / why we need it:\r\nAdds a log line for CertificateSigningRequests created...\r\n(중략)",
  "labels": [
    { "name": "sig/api-machinery", "color": "d2b48c" },
    { "name": "size/XS", "color": "009900" },
    { "name": "kind/feature", "color": "c7def8" },
    { "name": "sig/auth", "color": "d2b48c" },
    "..." 
  ],
  "created_at": "2026-02-09T15:20:15Z",
  "updated_at": "2026-02-09T19:55:13Z",
  "closed_at": "2026-02-09T19:53:15Z",
  "merged_at": null,
  "head": {
    "label": "youwalther65:csr-create",
    "ref": "csr-create",
    "sha": "873c3ca2"
  },
  "base": {
    "label": "kubernetes:master",
    "ref": "master",
    "repo": {
      "full_name": "kubernetes/kubernetes",
      "description": "Production-Grade Container Scheduling and Management"
    }
  },
  "requested_reviewers": [
    { "login": "dims" },
    { "login": "s-urbaniak" }
  ],
  "_links": {
    "self": { "href": ".../pulls/136885" },
    "html": { "href": ".../pull/136885" }
  },
  "author_association": "NONE"
}
```

In [0]:
import pandas as pd
import re

def extract_pr_features_one_row(pr: dict) -> dict:
    """
    ```python
    df = pd.DataFrame([extract_pr_features_one_row(pr) for pr in raw_prs])
    print(df.head())
    ```
    """
    body = pr.get("body") or ""
    labels = [l["name"] for l in pr.get("labels", [])]

    return {
        # --- Meta
        "pr_number": pr.get("number"),
        "state": pr.get("state"),
        "draft": pr.get("draft"),
        "created_at": pr.get("created_at"),
        "closed_at": pr.get("closed_at"),
        "merged_at": pr.get("merged_at"),

        # --- Author / Process
        "author_login": pr.get("user", {}).get("login"),
        "author_association": pr.get("author_association"),
        "requested_reviewers_n": len(pr.get("requested_reviewers", [])),
        "labels_n": len(labels),

        # --- Labels (Governance signals)
        "has_kind_feature": any(l.startswith("kind/feature") for l in labels),
        "has_kind_bug": any(l.startswith("kind/bug") for l in labels),
        "has_size_label": any(l.startswith("size/") for l in labels),
        "size_label": next((l.replace("size/", "") for l in labels if l.startswith("size/")), None),
        "has_needs_ok_to_test": "needs-ok-to-test" in labels,
        "has_do_not_merge": any(l.startswith("do-not-merge") for l in labels),

        # --- Body (NLP core)
        "body_raw": body,
        "body_len_chars": len(body),
        "body_len_lines": body.count("\n") + 1,
        "has_release_note": "release note" in body.lower(),
        "has_kind_declared": "/kind" in body.lower(),
        "has_issue_link": bool(re.search(r"#\d+", body)),

        # --- Branch
        "base_branch": pr.get("base", {}).get("ref"),
        "head_branch": pr.get("head", {}).get("ref"),
        "merge_commit_sha": pr.get("merge_commit_sha"),
    }



## PR 데이터 수집 시작

In [0]:
import asyncio

catalog = '2dt_final_databricks'

##############################################
#           Kubernetes 오픈소스에 대해
#           Pull Request 수집 및 저장
#
#   Kubernetes PR: https://github.com/kubernetes/kubernetes/pulls?q=is%3Apr+is%3Aclosed
#       => 86,000개의 Closed PR이 존재
#       => 이 중에서 master 브랜치에 대해서만 필터
#       => 만약 MAX_PAGE = 200 지정시 약 2만개의 PR 수집
##############################################
raw_prs = None

def collect_pr_data_of_kubernetes():
    nonlocal raw_prs
    fetcher = GitHubPRFetcher(github_api_config, repo_name='kubernetes')

    # # Fetch PR 기본 데이터
    MAX_PAGE = 200
    raw_prs = await fetcher.fetch_prs(
        state="closed",
        max_pages=MAX_PAGE
    )
    print(f"Fetched PRs: {len(raw_prs)}")

collect_pr_data_of_kubernetes()

# Dataframe으로 변환
df = spark.createDataFrame([extract_pr_features_one_row(pr) for pr in raw_prs])

print(df.head())
print(df.display())

# 테이블로 덮어쓰기 (기존 것 삭제 주의)
(
    df
    .filter(df.base_branch == "master")
    .withColumnRenamed("login", "author")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.github.k8s_prs")
)

fetching: 10 pages over...
fetching: 20 pages over...
fetching: 30 pages over...
fetching: 40 pages over...
fetching: 50 pages over...
fetching: 60 pages over...
fetching: 70 pages over...
fetching: 80 pages over...
fetching: 90 pages over...
fetching: 100 pages over...
fetching: 110 pages over...
fetching: 120 pages over...
fetching: 130 pages over...
fetching: 140 pages over...
fetching: 150 pages over...
fetching: 160 pages over...
fetching: 170 pages over...
fetching: 180 pages over...
fetching: 190 pages over...
fetching: 200 pages over...
Fetched PRs: 20000
Row(author_association='NONE', author_login='youwalther65', base_branch='master', body_len_chars=1315, body_len_lines=28, body_raw='#### What type of PR is this?\r\n\r\n/kind feature\r\n#### What this PR does / why we need it:\r\nAdds a log line for CertificateSigningRequests created, which was previously only visible in control plane `Audit` ogs.\r\n\r\n#### Which issue(s) this PR is related to:\r\n\r\nImplements #[136852](ht

author_association author_login base_branch body_len_chars body_len_lines body_raw closed_at created_at draft has_do_not_merge has_issue_link has_kind_bug has_kind_declared has_kind_feature has_needs_ok_to_test has_release_note has_size_label head_branch labels_n merge_commit_sha merged_at pr_number requested_reviewers_n size_label state NONE youwalther65 master 1315 28 #### What type of PR is this?

/kind feature
#### What this PR does / why we need it:
Adds a log line for CertificateSigningRequests created, which was previously only visible in control plane `Audit` ogs.

#### Which issue(s) this PR is related to:

Implements #[136852](https://github.com/kubernetes/kubernetes/issues/136852)

#### Special notes for your reviewer:

#### Does this PR introduce a user-facing change?
Adds another log line when a CertificateSigningRequests is created.

Example kubelet log (default `log-level = 2`):
`# journalctl -u kubelet | grep signing`
…
`Feb 09 13:24:57 .compute.internal kubelet[21230]: I0209 13:24:57.692781   21230 csr.go:86] "Certificate signing request created" logger="kubernetes.io/kubelet-serving" csr="csr-snjsj"`

Already existing kubelet log lines without PR:
`# journalctl -u kubelet | grep signing`
…
`Feb 09 13:25:12 .compute.internal kubelet[21230]: I0209 13:25:12.839312   21230 csr.go:276] "Certificate signing request is approved, waiting to be issued" logger="kubernetes.io/kubelet-serving" csr="csr-snjsj"`
`Feb 09 13:25:12 .compute.internal kubelet[21230]: I0209 13:25:12.851349   21230 csr.go:272] "Certificate signing request is issued" logger="kubernetes.io/kubelet-serving" csr="csr-snjsj"`


 2026-02-09T19:53:15Z 2026-02-09T15:20:15Z false true false false true true true false true csr-create 11 71f71d9b16035e44168e81a77171141a3f15abaf null 136885 2 XS closed CONTRIBUTOR atombrella master 580 25 #### What type of PR is this?

There are two cases that [modernize/minmax](https://pkg.go.dev/golang.org/x/tools/go/analysis/passes/modernize#hdr-Analyzer_minmax) caught, so I have included them in this batch.

/kind cleanup

#### What this PR does / why we need it:

#### Which issue(s) this PR is related to:

#### Special notes for your reviewer:

/assign @pohly

#### Does this PR introduce a user-facing change?
```release-note
NONE
```

#### Additional documentation e.g., KEPs (Kubernetes Enhancement Proposals), usage docs, etc.:

```docs
NONE
```
 2026-02-08T19:23:08Z 2026-02-08T13:35:40Z false false false false true false false false true feature/fmt_sprintf_unneeded 18 918b5ac90ba2e111b899257965a62d2ab19cb105 2026-02-08T19:23:08Z 136840 2 M closed MEMBER dims master 6366 149 Bump OpenTelemetry dependencies:
```
- go.opentelemetry.io/otel v1.39.0 → v1.40.0
- go.opentelemetry.io/otel/metric v1.39.0 → v1.40.0
- go.opentelemetry.io/otel/sdk v1.39.0 → v1.40.0
- go.opentelemetry.io/otel/trace v1.39.0 → v1.40.0
- go.opentelemetry.io/otel/exporters/otlp/otlptrace v1.39.0 → v1.40.0
- go.opentelemetry.io/otel/exporters/otlp/otlptrace/otlptracegrpc v1.39.0 → v1.40.0
- go.opentelemetry.io/contrib/.../otelrestful v0.64.0 → v0.65.0
- go.opentelemetry.io/contrib/.../otelhttp v0.64.0 → v0.65.0
- go.opentelemetry.io/contrib/.../otelgrpc v0.63.0 → v0.65.0
```

 *opentelemetry-go (v1.39.0 → v1.40.0)*

 Added:
 - AlwaysRecord sampler for trace sampling in SDK
 - Enabled() method on all synchronous metric instruments — lets callers skip expensive measurement prep when no pipeline is listening
 - semconv/v1.39.0 package with updated semantic conventions

 Performance:
 - 4x improvement for HistogramReservoir and FixedSizeReservoir exemplar handling
 - Faster concurrent histogram, synchronous gauge, and exponential histogram measurements

 Fixed:
 - DroppedAttributes count no longer includes non-attribute key-value pairs
 - SetAttributes no longer incorrectly logs attribute drops when none occurred
 - HTTP trace exporter now sets request.GetBody for proper HTTP/2 G

None


### 코멘트 데이터 가져오기
위에 있는 코드의 경우 PR 몸통 데이터를 가져온다고 생각하면 된다.  

그리고 PR에 같이 딸려있는 댓글(Comment)와 리뷰(Review)데이터도 API로 불러오게 된다.  

Review는 댓글과 별개로, 마지막으로 승인권자(Reviewer)가 남기는 내용(Content)와 기타 메타 정보가 포함된다.  



In [0]:
GITHUB_API = "https://api.github.com"

async def fetch_pr_discussions(client,owner,repo,pr_number,token,index=None):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    urls = {
        "issue_comments": f"{GITHUB_API}/repos/{owner}/{repo}/issues/{pr_number}/comments",
        # "review_comments": f"{GITHUB_API}/repos/{owner}/{repo}/pulls/{pr_number}/comments",
        "reviews": f"{GITHUB_API}/repos/{owner}/{repo}/pulls/{pr_number}/reviews",
    }

    async def _get(name, url):
        resp = await client.get(url, headers=headers)
        resp.raise_for_status()

        remaining = int(resp.headers.get("X-RateLimit-Remaining", 0))
        if remaining < 50:
            await asyncio.sleep(30)

        return name, resp.json()

    tasks = [_get(name, url) for name, url in urls.items()]
    results = await asyncio.gather(*tasks)

    if index and index % 100 == 0:
        print(f"✅ fetched PR #{pr_number} ({index} done)")

    return {
        "pr_number": pr_number,
        "issue_comments": dict(results)["issue_comments"],
        # "review_comments": dict(results)["review_comments"],
        "reviews": dict(results)["reviews"],
    }

async def bounded_fetch(index, semaphore, client, owner, repo, pr_number, token):
    async with semaphore:
        return await fetch_pr_discussions(
            client, owner, repo, pr_number, token, index=index
        )

async def fetch_all_discussions(pr_rows, token, concurrency=10, owner=None, repo=None, max_tasks=None):
    semaphore = asyncio.Semaphore(concurrency)

    async with httpx.AsyncClient(timeout=30.0) as client:
        tasks = [
            bounded_fetch(
                i,
                semaphore,
                client,
                owner or r["owner"],
                repo or r["repo"],
                r["pr_number"],
                token
            )
            for i, r in enumerate(pr_rows)
            if not max_tasks or (max_tasks and i+1 <= max_tasks)
        ]

        results = await asyncio.gather(*tasks, return_exceptions=True)
        return results

# 이슈 코멘트 저장
def save_to_table_as_issue_comments(result_df):
    issue_comment_rows = []

    for r in result_df:
        if isinstance(r, Exception):
            print(r)
            continue

        pr_number = r["pr_number"]

        for c in r.get("issue_comments", []):
            user = c.get("user") or {}
            reactions = c.get("reactions") or {}

            body = c.get("body", "") or ""

            created_at = c.get("created_at")
            updated_at = c.get("updated_at")

            is_edited = int(created_at != updated_at)

            issue_comment_rows.append({
                # identifiers
                "comment_id": c.get("id"),
                "pr_number": pr_number,

                # user signals
                "author": user.get("login"),
                "user_type": user.get("type"),              # User / Bot
                "is_bot": int(user.get("type") == "Bot"),
                "is_site_admin": int(user.get("site_admin", False)),
                "author_association": c.get("author_association"),

                # text
                "body": body,
                "body_length": len(body),

                # time
                "created_at": created_at,
                "updated_at": updated_at,
                "is_edited": is_edited,

                # reactions (social signal)
                "reaction_total": reactions.get("total_count", 0),
                "reaction_plus1": reactions.get("+1", 0),
                "reaction_minus1": reactions.get("-1", 0),
                "reaction_eyes": reactions.get("eyes", 0),
                "reaction_heart": reactions.get("heart", 0),
            })

    return issue_comment_rows

# 리뷰 저장
def save_to_table_as_review(result_df):
    review_rows = []

    for r in result_df:
        if isinstance(r, Exception):
            continue

        pr_number = r["pr_number"]

        for rv in r.get("reviews", []):
            user = rv.get("user") or {}
            body = rv.get("body", "") or ""

            review_rows.append({
                # identifiers
                "review_id": rv.get("id"),
                "pr_number": pr_number,

                # reviewer identity
                "author": user.get("login"),
                "user_type": user.get("type"),             # User / Bot
                "is_bot": int(user.get("type") == "Bot"),
                "is_site_admin": int(user.get("site_admin", False)),
                "author_association": rv.get("author_association"),

                # decision signal
                "state": rv.get("state"),          
                "is_approved": int(rv.get("state") == "APPROVED"),
                "is_changes_requested": int(rv.get("state") == "CHANGES_REQUESTED"),

                # text
                "body": body,
                "body_length": len(body),

                # time
                "submitted_at": rv.get("submitted_at"),
            })

    return review_rows

#---------------------------------------
#
#           코드 실행
#
#---------------------------------------

_exists_df = spark.table("2dt_final_databricks.github.k8s_prs")
pr_rows = _exists_df.collect()
print(f"Rows number: {len(pr_rows)}")
results = await fetch_all_discussions(
    pr_rows, 
    GITHUB_TOKEN,
    concurrency=10,
    owner="kubernetes",
    repo="kubernetes",
    max_tasks=None
)

print(f"Fetched Discussions: {len(results)}")


################## 테스트시 드랍###################
# spark.sql("DROP TABLE IF EXISTS 2dt_final_databricks.github.k8s_prs_issue_comments")
# spark.sql("DROP TABLE IF EXISTS 2dt_final_databricks.github.k8s_prs_reviews")
################################################


issue_results = save_to_table_as_issue_comments(results)
review_results = save_to_table_as_review(results)

print(f"Issue Comments: {len(issue_results)}")
print(f"Reviews: {len(review_results)}")

issue_comments_df = spark.createDataFrame(issue_results)
reviews_df = spark.createDataFrame(review_results)

print("Created spark dataframes.")

print("Checking table exsits...")
spark.sql("""
CREATE TABLE IF NOT EXISTS 2dt_final_databricks.github.k8s_prs_issue_comments (
  comment_id           BIGINT,
  pr_number            BIGINT,

  author               STRING,
  user_type            STRING,
  is_bot               INT,
  is_site_admin        INT,
  author_association   STRING,

  body                 STRING,
  body_length          INT,

  created_at           TIMESTAMP,
  updated_at           TIMESTAMP,
  is_edited            INT,

  reaction_total       INT,
  reaction_plus1       INT,
  reaction_minus1      INT,
  reaction_eyes        INT,
  reaction_heart       INT
)
USING DELTA;
"""
)
spark.sql("""
CREATE TABLE IF NOT EXISTS 2dt_final_databricks.github.k8s_prs_reviews (
  review_id             BIGINT,
  pr_number             BIGINT,

  author                STRING,
  user_type             STRING,
  is_bot                INT,
  is_site_admin         INT,
  author_association    STRING,

  state                 STRING,
  is_approved           INT,
  is_changes_requested  INT,

  body                  STRING,
  body_length           INT,

  submitted_at          TIMESTAMP
)
USING DELTA;
        
"""
)

print("Creating temp view...")
issue_comments_df.createOrReplaceTempView("tmp_k8s_prs_issue_comments")
reviews_df.createOrReplaceTempView("tmp_k8s_prs_reviews")

print("Merging into table...")
spark.sql("""
MERGE INTO `2dt_final_databricks`.github.k8s_prs_issue_comments t
USING tmp_k8s_prs_issue_comments s
ON t.comment_id = s.comment_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
MERGE INTO `2dt_final_databricks`.github.k8s_prs_reviews t
USING tmp_k8s_prs_reviews s
ON t.review_id = s.review_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

Rows number: 17499
[PR 136885] fetched (0)
[PR 136638] fetched (50)
[PR 136516] fetched (100)
[PR 136407] fetched (150)
[PR 136292] fetched (200)
[PR 136180] fetched (250)
[PR 136067] fetched (300)
[PR 135940] fetched (350)
[PR 135821] fetched (400)
[PR 135695] fetched (450)
[PR 135600] fetched (500)
[PR 135492] fetched (550)
[PR 135377] fetched (600)
[PR 135263] fetched (650)
[PR 135161] fetched (700)
[PR 135080] fetched (750)
[PR 134980] fetched (800)
[PR 118614] fetched (12650)
[PR 118537] fetched (12700)
[PR 118458] fetched (12750)
[PR 118388] fetched (12800)
[PR 118311] fetched (12850)
[PR 118223] fetched (12900)
[PR 118130] fetched (12950)
[PR 118048] fetched (13000)
[PR 117982] fetched (13050)
[PR 117912] fetched (13100)
[PR 114273] fetched (13150)
[PR 114180] fetched (13200)
[PR 114078] fetched (13250)
[PR 113985] fetched (13300)
[PR 113879] fetched (13350)
[PR 113793] fetched (13400)
[PR 113698] fetched (13450)
[PR 113622] fetched (13500)
[PR 113546] fetched (13550)
[PR 113477

Client error '403 Forbidden' for url 'https://api.github.com/repos/kubernetes/kubernetes/pulls/110904/reviews'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Client error '403 Forbidden' for url 'https://api.github.com/repos/kubernetes/kubernetes/issues/110902/comments'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Client error '403 Forbidden' for url 'https://api.github.com/repos/kubernetes/kubernetes/issues/110901/comments'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Client error '403 Forbidden' for url 'https://api.github.com/repos/kubernetes/kubernetes/issues/110900/comments'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
Client error '403 Forbidden' for url 'https://api.github.com/repos/kubernetes/kubernetes/pulls/110899/reviews'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Statu

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## 실버 테이블 생성


In [0]:
## 실버 테이블

from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, size,
    length, when, dayofweek, hour
)

print(f"Catalog : {catalog}")
silver_df = (
    df
    # ── 시간 기반 Feature ─────────────────
    .withColumn(
        "time_to_close_hours",
        (unix_timestamp(to_timestamp("closed_at")) - unix_timestamp(to_timestamp("created_at"))) / 3600
    )
    .withColumn(
        "time_to_merge_hours",
        when(
            col("merged_at").isNotNull(),
            (unix_timestamp(to_timestamp("merged_at")) - unix_timestamp(to_timestamp("created_at"))) / 3600
        )
    )
    # ── Boolean → Numeric ─────────────────
    .withColumn("is_merged", col("merged_at").isNotNull().cast("int"))
    .withColumn("is_draft", col("draft").cast("int"))
)
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.github.k8s_prs_silver")
)

# 품질 체크
spark.sql(f"""
SELECT
  count(*) AS total,
  avg(time_to_close_hours) AS avg_lifetime,
  avg(time_to_merge_hours) AS avg_merge_time
FROM `2dt_final_databricks`.github.k8s_prs_silver
""").show()

Catalog : 2dt_final_databricks
+-----+------------------+-----------------+
|total|      avg_lifetime|   avg_merge_time|
+-----+------------------+-----------------+
|20000|1181.9451320833336|573.1500590530095|
+-----+------------------+-----------------+

